# Phân tích thực nghiệm Gomoku (Caro) 9×9 — Level 1 đến Level 5

Notebook này đọc trực tiếp các tệp nhật ký JSONL trong `logs/` để tổng hợp đầy đủ số liệu và hình minh họa cho báo cáo cuối kỳ:
1. Bảng kết quả Thắng / Hòa / Thua, điểm chuẩn hóa `score = (wins + 0.5 * draws) / games`, khoảng tin cậy **Wilson 95%**, thời gian trung bình/nước, độ trễ **P95** và khối lượng duyệt (`nodes` / `simulations`) của cả 5 agent (`random`, `heuristic`, `alphabeta`, `mcts`, `cnn`).
2. Ma trận đối đầu trực tiếp (Head-to-Head Heatmap) ở hai chế độ ngân sách `compute` và `time` (100 ván/cặp × 10 cặp = 1.000 ván/chế độ).
3. Phân tích lợi thế đi trước (`X` so với `O`), phân bố độ dài ván cờ, đường cong huấn luyện `cnn` Level 5 (`train_cnn.jsonl`) và nghiên cứu cắt giảm **Ablation 1** (`ablation_search.jsonl`).

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from arena.round_robin import (
    format_matrix,
    format_summary,
    head_to_head,
    side_advantage,
    summarize,
)

LOG_DIR = ROOT / "logs"
FIG_DIR = ROOT / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

AGENTS = ["random", "heuristic", "alphabeta", "mcts", "cnn", "mcts_cnn"]

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

runs = {}
for label, filename in (("compute", "rr_compute_100.jsonl"), ("time", "rr_time_100.jsonl")):
    recs = load_jsonl(LOG_DIR / filename)
    if recs:
        runs[label] = recs

if not runs and (LOG_DIR / "round_robin.jsonl").exists():
    runs["compute"] = load_jsonl(LOG_DIR / "round_robin.jsonl")

summaries = {}
for mode, recs in runs.items():
    present = [a for a in AGENTS if any(r["agent_x"] == a or r["agent_o"] == a for r in recs)]
    summaries[mode] = summarize(recs, agent_names=present)
    adv = side_advantage(recs)
    print(f"=== Chế độ ngân sách: {mode} ({len(recs)} ván) ===")
    print(format_summary(summaries[mode]))
    print("\nMa trận đối đầu (score | W-D-L):")
    print(format_matrix(head_to_head(recs, present), present))
    print(
        f"\nLợi thế đi trước: X thắng {adv['x_wins']}/{adv['games']} ({adv['x_win_rate']:.1%}), "
        f"O thắng {adv['o_wins']}/{adv['games']} ({adv['o_win_rate']:.1%}), "
        f"Hòa {adv['draws']}/{adv['games']} ({adv['draw_rate']:.1%})\n"
    )

=== Chế độ ngân sách: compute (1000 ván) ===
Agent         Thắng    Hòa   Thua    Điểm          CI 95%   Thời gian/nước (s)    P95 (s)  Nodes/Sims
-----------------------------------------------------------------------------------------------------
random            0      0    400   0.000    [0.00, 0.01]             0.000012   0.000025         0.0
heuristic       245      9    146   0.624    [0.58, 0.67]             0.001299   0.002845         0.0
alphabeta       360     28     12   0.935    [0.91, 0.96]             0.051666   0.110720       157.1
mcts            101      0    299   0.253    [0.21, 0.30]             0.054437   0.086779       400.0
cnn             258     35    107   0.689    [0.64, 0.73]             0.001168   0.001886         0.0

Ma trận đối đầu (score | W-D-L):
A \ B                   random         heuristic         alphabeta              mcts               cnn
------------------------------------------------------------------------------------------------------
r

In [2]:
modes = list(runs.keys())
fig, axes = plt.subplots(1, len(modes), figsize=(6.8 * len(modes), 5.6), squeeze=False, layout="constrained")

for idx, mode in enumerate(modes):
    ax = axes[0, idx]
    names = list(summaries[mode].keys())
    h2h = head_to_head(runs[mode], names)
    n = len(names)
    grid = np.full((n, n), np.nan)
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i != j:
                grid[i, j] = h2h[a][b]["score"]

    im = ax.imshow(grid, vmin=0.0, vmax=1.0, cmap="RdYlGn")
    ax.set_xticks(range(n), names)
    ax.set_yticks(range(n), names)
    ax.set_xlabel("Đối thủ (cột B)")
    ax.set_ylabel("Tác tử đánh giá (hàng A)")
    ax.set_title(f"Ma trận đối đầu — {mode} ({len(runs[mode])} ván)")

    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center", fontweight="bold")
            else:
                c = h2h[a][b]
                txt = f"{c['score']:.2f}\n({c['wins']}-{c['draws']}-{c['losses']})"
                ax.text(j, i, txt, ha="center", va="center", fontsize=8.5)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="Điểm chuẩn hóa (score)")
out_h2h = FIG_DIR / "head_to_head_matrix.png"
fig.savefig(out_h2h, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_h2h)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\head_to_head_matrix.png


In [3]:
names = list(summaries[modes[0]].keys())
fig, (ax_score, ax_time) = plt.subplots(1, 2, figsize=(12.5, 4.7))
x = np.arange(len(names))
width = 0.36 if len(modes) > 1 else 0.5

for idx, mode in enumerate(modes):
    offset = (idx - (len(modes) - 1) / 2) * width
    s_map = summaries[mode]
    scores = [s_map[a]["score"] for a in names]
    err_low = [s_map[a]["score"] - s_map[a]["ci_low"] for a in names]
    err_high = [s_map[a]["ci_high"] - s_map[a]["score"] for a in names]
    ax_score.bar(x + offset, scores, width, yerr=[err_low, err_high], capsize=4, label=f"mode={mode}")

    means_ms = [s_map[a]["time_per_move"] * 1000.0 for a in names]
    p95_ms = [s_map[a]["p95_time"] * 1000.0 for a in names]
    ax_time.bar(x + offset, means_ms, width, label=f"TB ({mode})")
    ax_time.scatter(x + offset, p95_ms, marker="D", zorder=5, label=f"P95 ({mode})")

ax_score.set_xticks(x, names)
ax_score.set_ylim(0.0, 1.08)
ax_score.set_ylabel("Score = (Thắng + 0.5 × Hòa) / Số ván")
ax_score.set_title("Điểm thi đấu & Khoảng tin cậy Wilson 95%")
ax_score.grid(axis="y", alpha=0.3)
ax_score.legend()

ax_time.set_xticks(x, names)
ax_time.set_yscale("log")
ax_time.set_ylabel("Thời gian mỗi nước (ms, thang log)")
ax_time.set_title("Chi phí suy luận trung bình và độ trễ P95")
ax_time.grid(axis="y", alpha=0.3)
ax_time.legend()

fig.tight_layout()
out_perf = FIG_DIR / "score_and_latency.png"
fig.savefig(out_perf, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_perf)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\score_and_latency.png


In [4]:
primary_mode = modes[0]
primary_recs = runs[primary_mode]
adv = side_advantage(primary_recs)
lengths = [len(r["moves"]) for r in primary_recs]

fig, (ax_side, ax_len) = plt.subplots(1, 2, figsize=(11.5, 4.3))

labels = ["X thắng (đi trước)", "O thắng (đi sau)", "Hòa"]
counts = [adv["x_wins"], adv["o_wins"], adv["draws"]]
bars = ax_side.bar(labels, counts, color=["#d91e18", "#222222", "#8c8c8c"])
for b, c in zip(bars, counts):
    pct = (c / adv["games"] * 100.0) if adv["games"] else 0.0
    ax_side.text(b.get_x() + b.get_width() / 2, c + max(1, adv["games"] * 0.01), f"{c} ({pct:.1f}%)", ha="center")
ax_side.set_ylim(0, max(counts) * 1.18 + 1)
ax_side.set_ylabel("Số ván")
ax_side.set_title(f"Tác động của lượt đi trước ({primary_mode}, N={adv['games']})")

ax_len.hist(lengths, bins=range(min(lengths), max(lengths) + 2), edgecolor="black", alpha=0.8)
ax_len.axvline(np.mean(lengths), color="red", linestyle="--", label=f"TB = {np.mean(lengths):.1f} nước")
ax_len.set_xlabel("Tổng số nước đi mỗi ván")
ax_len.set_ylabel("Số ván")
ax_len.set_title("Phân bố độ dài ván đấu trên bàn 9×9")
ax_len.legend()

fig.tight_layout()
out_side = FIG_DIR / "side_advantage_and_length.png"
fig.savefig(out_side, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_side)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\side_advantage_and_length.png


In [5]:
train_recs = [r for r in load_jsonl(LOG_DIR / "train_cnn.jsonl") if r.get("event") == "epoch"]
abl_recs = [r for r in load_jsonl(LOG_DIR / "ablation_search.jsonl") if r.get("phase") == "node_benchmark"]

if train_recs and abl_recs:
    fig, (ax_loss, ax_acc, ax_abl) = plt.subplots(1, 3, figsize=(15, 4.3))
    ep = [r["epoch"] for r in train_recs]
    ax_loss.plot(ep, [r["train_loss"] for r in train_recs], label="Train loss", marker="o", markersize=3)
    ax_loss.plot(ep, [r["val_loss"] for r in train_recs], label="Val loss", marker="s", markersize=3)
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.set_title("Đường cong Loss huấn luyện CNN (Level 5)")
    ax_loss.grid(alpha=0.3)
    ax_loss.legend()

    ax_acc.plot(ep, [r["train_acc"] * 100 for r in train_recs], label="Train Top-1", marker="o", markersize=3)
    ax_acc.plot(ep, [r["val_top1_acc"] * 100 for r in train_recs], label="Val Top-1", marker="s", markersize=3)
    ax_acc.plot(ep, [r["val_top3_acc"] * 100 for r in train_recs], label="Val Top-3", marker="^", markersize=3)
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Độ chính xác (%)")
    ax_acc.set_title("Độ chính xác dự đoán nước chuyên gia")
    ax_acc.grid(alpha=0.3)
    ax_acc.legend()

    cfg_labels = [r["config"].replace("_", "\n") for r in abl_recs]
    nodes_vals = [r["mean_nodes"] for r in abl_recs]
    bars = ax_abl.bar(cfg_labels, nodes_vals, color=["#c92a2a", "#f08c00", "#2b8a3e", "#1971c2"])
    for b, v in zip(bars, nodes_vals):
        ax_abl.text(b.get_x() + b.get_width() / 2, v + 20, f"{v:.0f}", ha="center", fontsize=8.5)
    ax_abl.set_ylim(0, max(nodes_vals) * 1.16)
    ax_abl.set_ylabel("Số nút duyệt TB / nước")
    ax_abl.set_title("Ablation 1: Hiệu quả cắt tỉa Alpha-Beta")
    ax_abl.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    out_learn = FIG_DIR / "cnn_learning_and_ablation1.png"
    fig.savefig(out_learn, dpi=160, bbox_inches="tight")
    print("Đã lưu:", out_learn)
    plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\cnn_learning_and_ablation1.png
